# Cluster Validation

In [ ]:
from pathlib import Path

from cluster_validation import ClusterValidationConfig, run_cluster_validation
from cluster_validation.viz import plot_all

ROOT = Path().resolve().parents[0]

## Config

In [ ]:
# --- config ---
H5AD_DIR   = ROOT / "data/scbasecount/2026-01-12/h5ad/GeneFull/Homo_sapiens"
SUMMARY    = ROOT / "output/metadata/datasets.csv"
OUTPUT_DIR = ROOT / "tmp/data"
FIGS_DIR   = ROOT / "tmp/figs"

accessions = sorted(p.stem for p in H5AD_DIR.glob("*.h5ad"))

## Run

In [ ]:
results = {}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for srx in accessions:
    cfg = ClusterValidationConfig(
        srxAccession=srx,
        summaryPath=SUMMARY,
        localH5adRoot=H5AD_DIR,
        outputDir=OUTPUT_DIR,
    )
    adata, result = run_cluster_validation(cfg)
    plot_all(adata, result, figs_dir=FIGS_DIR / srx)
    results[srx] = result

## Results Summary

In [ ]:
import pandas as pd

rows = [
    {
        "srx": r.srxAccession,
        "resolution": r.selectedResolution,
        "n_pcs": r.nPcs,
        "cells_kept": r.kFiltered,
        "clusters_pre": r.nClustersPreMerge,
        "clusters_post": r.nClustersPostMerge,
        "adata_path": str(r.adataPath),
    }
    for r in results.values()
]
pd.DataFrame(rows)

In [ ]:
print(f"Wrote {len(results)} h5ad files to {OUTPUT_DIR}")